In [ ]:
from google.colab import files
import pandas as pd

# Patient & Encounter Metric Creation

# Upload required files: DimPatient.csv and FactEncounter.csv
uploaded = files.upload()

# Load datasets
dim_patient = pd.read_csv("DimPatient.csv")
fact_encounter = pd.read_csv("FactEncounter.csv")

# Clean column names
dim_patient.columns = dim_patient.columns.str.lower().str.strip().str.replace(" ", "_")
fact_encounter.columns = fact_encounter.columns.str.strip().str.replace(" ", "_")

# Convert date columns
dim_patient["birth_date"] = pd.to_datetime(dim_patient["birth_date"], errors="coerce")
fact_encounter["CheckinDate"] = pd.to_datetime(fact_encounter["CheckinDate"], errors="coerce")
fact_encounter["CheckoutDate"] = pd.to_datetime(fact_encounter["CheckoutDate"], errors="coerce")

# Create patient age
today = pd.Timestamp.today()
dim_patient["age"] = ((today - dim_patient["birth_date"]).dt.days // 365)

# Create age groups
dim_patient["age_group"] = pd.cut(
    dim_patient["age"],
    bins=[0, 18, 35, 50, 65, 120],
    labels=["Child", "Young Adult", "Adult", "Middle-aged", "Senior"]
)

# Create BMI
dim_patient["bmi"] = (
    dim_patient["weight"] / ((dim_patient["height"] / 100) ** 2)
).round(2)

# Create BMI category
dim_patient["bmi_category"] = pd.cut(
    dim_patient["bmi"],
    bins=[0, 18.5, 25, 30, 100],
    labels=["Underweight", "Normal", "Overweight", "Obese"]
)

# Create high-risk BMI flag
dim_patient["high_risk_bmi"] = (
    (dim_patient["bmi"] < 18.5) | (dim_patient["bmi"] >= 30)
).astype(int)

# Create length of stay
fact_encounter["LengthOfStay"] = (
    fact_encounter["CheckoutDate"] - fact_encounter["CheckinDate"]
).dt.days

# Create length of stay category
fact_encounter["LOS_Category"] = pd.cut(
    fact_encounter["LengthOfStay"],
    bins=[-1, 1, 3, 7, 100],
    labels=["Same Day", "Short", "Medium", "Long"]
)

# Create radiology and endoscopy usage flags
fact_encounter["HasRadiology"] = (
    fact_encounter["RadiologyProcedureCount"] > 0
).astype(int)

fact_encounter["HasEndoscopy"] = (
    fact_encounter["EndoscopyProcedureCount"] > 0
).astype(int)

# Create total procedures column
fact_encounter["TotalProcedures"] = (
    fact_encounter["RadiologyProcedureCount"] +
    fact_encounter["EndoscopyProcedureCount"]
)

# Create admission timing metrics
fact_encounter["WeekendAdmission"] = (
    fact_encounter["CheckinDate"].dt.dayofweek >= 5
).astype(int)

fact_encounter["AdmissionMonth"] = fact_encounter["CheckinDate"].dt.month_name()
fact_encounter["AdmissionQuarter"] = fact_encounter["CheckinDate"].dt.quarter

# Save updated files
dim_patient.to_csv("DimPatient_with_metrics.csv", index=False)
fact_encounter.to_csv("FactEncounter_with_metrics.csv", index=False)

# Download updated files
files.download("DimPatient_with_metrics.csv")
files.download("FactEncounter_with_metrics.csv")

print("Patient and encounter metrics created successfully.")

Saving DimPatient.csv to DimPatient (1).csv
Saving FactEncounter.csv to FactEncounter (1).csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Patient and encounter metrics created successfully.
